In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
import numpy as np
from datetime import datetime
import random
from pymongo import MongoClient
from bson import ObjectId
import warnings

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# DATABASE + IB CONNECTION
# ══════════════════════════════════════════════════════════════════════════════

client            = MongoClient('mongodb://localhost:27017/')
db                = client['dipBuy_db']
trades_collection = db['trades_rsi']
exclude_collection = db['excluded_tickers_rsi']

util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))

tickers = [
    'WMT', 'MU', 'SAVA', 'SLDB', 'SLV', 'NEM', 'CTXR', 'ONON', 'IONQ', 'MARA',
    'MRVL', 'T', 'CCL', 'XYZ', 'PG', 'ONDS', 'NFLX', 'V', 'ADBE', 'MS', 'CXW',
    'BA', 'BABA', 'DAL', 'JPM', 'C', 'AMZN', 'UAL', 'BRK.B', 'IBM', 'MSFT', 'KO',
    'MSTR', 'COIN', 'GLD', 'META', 'AAL', 'OSCR', 'QSI', 'SPY', 'NVO', 'DIS',
    'AAPL', 'BMNR', 'GRAB', 'RBLX', 'AFRM', 'NVDA', 'FUBO', 'PYPL', 'GOOG', 'BTC',
    'RGTI', 'GPRO', 'OKLO', 'AMD', 'XPEV', 'SHEL', 'TSM', 'SNOW', 'NET', 'SES',
    'TSLA', 'BP', 'UBER', 'INTC', 'MRNA', 'IREN', 'ORCL', 'HIMS', 'NBIS'
]
contracts = [Stock(t, 'SMART', 'USD') for t in tickers]
ib.qualifyContracts(*contracts)


# ══════════════════════════════════════════════════════════════════════════════
# MONGO SANITIZER
# ══════════════════════════════════════════════════════════════════════════════

def sanitize_for_mongo(d):
    """Recursively convert numpy types to native Python types."""
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out[k] = sanitize_for_mongo(v)
        elif hasattr(v, 'item'):
            out[k] = v.item()
        else:
            out[k] = v
    return out


# ══════════════════════════════════════════════════════════════════════════════
# INDICATORS
# ══════════════════════════════════════════════════════════════════════════════

def calc_rsi(df, period=14):
    d    = df['close'].diff()
    gain = d.clip(lower=0).ewm(com=period - 1, min_periods=period).mean()
    loss = (-d.clip(upper=0)).ewm(com=period - 1, min_periods=period).mean()
    df['RSI'] = 100 - 100 / (1 + gain / (loss + 1e-10))
    return df


# ══════════════════════════════════════════════════════════════════════════════
# MONGO HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def create_trade_doc(symbol, entry_price, qty, rsi_at_entry):
    return {
        'instrument':      symbol,
        'direction':       'long',
        'entry_price':     float(entry_price),
        'quantity':        int(qty),
        'remaining_qty':   int(qty),
        'rsi_at_entry':    float(rsi_at_entry),
        'entry_timestamp': datetime.now(),
        'order_id':        str(ObjectId()),
        'exit_price':      None,
        'exit_timestamp':  None,
        'exit_rsi':        None,
        'reason':          None,
        'realized_pnl':    None,
        'status':          'open',
        'created_at':      datetime.now(),
        'updated_at':      datetime.now(),
    }

def db_update(trade_id, updates):
    updates['updated_at'] = datetime.now()
    trades_collection.update_one({'_id': ObjectId(trade_id)}, {'$set': updates})

def do_sell(contract, qty, entry_price, sell_price, rsi, reason, trade_id):
    """Execute market sell and persist to MongoDB."""
    ib.placeOrder(contract, MarketOrder('SELL', qty))
    pnl = (sell_price - entry_price) * qty
    db_update(trade_id, {
        'exit_price':     float(sell_price),
        'exit_timestamp': datetime.now(),
        'exit_rsi':       float(rsi),
        'reason':         reason,
        'realized_pnl':   float(pnl),
        'status':         'closed',
    })
    sign = '+' if pnl >= 0 else ''
    print(f"  ✅ SELL {qty}sh @ ${sell_price:.2f}  RSI={rsi:.1f}  [{reason}]  PnL: {sign}${pnl:.2f}")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

async def check_and_trade():
    """
    ╔══════════════════════════════════════════════════════════════════════════╗
    ║  SIMPLE RSI DIP BUY STRATEGY                                            ║
    ╠══════════════════════════════════════════════════════════════════════════╣
    ║                                                                          ║
    ║  BUY  — RSI crosses BELOW 30 (oversold dip)                             ║
    ║         Trigger: previous RSI >= 30 AND current RSI < 30                ║
    ║                                                                          ║
    ║  SELL — RSI crosses ABOVE 70 (overbought)                               ║
    ║         Trigger: previous RSI <= 70 AND current RSI > 70                ║
    ║                                                                          ║
    ║  Position size: 40 shares fixed                                          ║
    ║  One trade per ticker per day                                            ║
    ║                                                                          ║
    ╚══════════════════════════════════════════════════════════════════════════╝
    """

    while True:
        ib.positions()  # refresh internal positions cache

        for contract in contracts:
            symbol = contract.symbol
            print(f"\n{'='*60}")
            print(f"  {symbol}  {datetime.now().strftime('%H:%M:%S')}")
            print(f"{'='*60}")

            # ── Fetch 5-min bars ──────────────────────────────────────────
            bars = ib.reqHistoricalData(
                contract,
                endDateTime='',
                durationStr='1 D',
                barSizeSetting='5 mins',
                whatToShow='TRADES',
                useRTH=False,
                formatDate=1
            )
            if not bars or len(bars) < 20:
                print(f"  No data / insufficient bars for {symbol}")
                continue

            df         = util.df(bars)
            df.columns = [c.lower() for c in df.columns]
            df         = calc_rsi(df)

            price    = float(df.iloc[-1]['close'])
            rsi_now  = float(df.iloc[-1]['RSI'])
            rsi_prev = float(df.iloc[-2]['RSI'])

            print(f"  Price=${price:.2f}  RSI={rsi_now:.1f}  RSI_prev={rsi_prev:.1f}")

            open_trade = trades_collection.find_one({'instrument': symbol, 'status': 'open'})

            # ══════════════════════════════════════════════════════════════
            # POSITION MANAGEMENT — SELL when RSI crosses above 70
            # ══════════════════════════════════════════════════════════════

            if open_trade:
                entry_price = float(open_trade['entry_price'])
                qty         = int(open_trade['remaining_qty'])
                trade_id    = open_trade['_id']
                current_pnl = (price - entry_price) * qty
                profit_pct  = (price - entry_price) / entry_price

                print(f"  OPEN  entry=${entry_price:.2f}  qty={qty}"
                      f"  P&L={profit_pct:+.2%}  (${current_pnl:+.2f})")

                # ── SELL — RSI crosses above 70 ───────────────────────────
                if rsi_prev <= 70 and rsi_now > 70:
                    do_sell(contract, qty, entry_price, price,
                            rsi_now, 'RSI_CROSSED_ABOVE_70', trade_id)
                    print(f"  📈 SELL triggered: RSI {rsi_prev:.1f} → {rsi_now:.1f} (crossed above 70)")
                else:
                    print(f"  Holding — waiting for RSI > 70 to sell  (RSI={rsi_now:.1f})")

            # ══════════════════════════════════════════════════════════════
            # ENTRY LOGIC — BUY when RSI crosses below 30
            # ══════════════════════════════════════════════════════════════

            else:
                today = datetime.now().date()
                if exclude_collection.find_one({'ticker': symbol, 'date': today.isoformat()}):
                    print(f"  Already traded {symbol} today — skipping.")
                    continue

                # ── BUY — RSI crosses below 30 ────────────────────────────
                if rsi_prev >= 30 and rsi_now < 30:
                    order_size = 40

                    ib.placeOrder(contract, MarketOrder('BUY', order_size))

                    doc = create_trade_doc(symbol, price, order_size, rsi_now)
                    trades_collection.insert_one(sanitize_for_mongo(doc))
                    exclude_collection.insert_one({'ticker': symbol, 'date': today.isoformat()})

                    print(f"  🚀 BUY  {symbol}")
                    print(f"     Entry:  ${price:.2f}")
                    print(f"     Shares: {order_size}")
                    print(f"     RSI:    {rsi_now:.1f}  (crossed below 30)")
                else:
                    print(f"  No entry — RSI={rsi_now:.1f}  (waiting for RSI to cross below 30)")

            await asyncio.sleep(0.5)  # brief pause between tickers

        print(f"\n{'='*60}")
        print(f"  Scan complete. Next scan in 5 minutes.")
        print(f"{'='*60}")
        await asyncio.sleep(300)


# ══════════════════════════════════════════════════════════════════════════════
# RUN
# ══════════════════════════════════════════════════════════════════════════════

try:
    ib.run(check_and_trade())
except Exception as e:
    print(f"Error: {e}")
    import traceback; traceback.print_exc()
finally:
    ib.disconnect()
    print("Disconnected from IB.")

Error 200, reqId 5: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')
Error 200, reqId 31: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')
Unknown contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')
Unknown contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')



  WMT  08:45:26
  Price=$122.82  RSI=50.6  RSI_prev=50.6
  No entry — RSI=50.6  (waiting for RSI to cross below 30)

  MU  08:45:27
  Price=$370.16  RSI=40.3  RSI_prev=40.3
  No entry — RSI=40.3  (waiting for RSI to cross below 30)


Error 200, reqId 76: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')



  SAVA  08:45:28
  No data / insufficient bars for SAVA

  SLDB  08:45:28
  Price=$7.09  RSI=0.0  RSI_prev=0.0
  No entry — RSI=0.0  (waiting for RSI to cross below 30)

  SLV  08:45:28
  Price=$61.28  RSI=45.3  RSI_prev=49.8
  No entry — RSI=45.3  (waiting for RSI to cross below 30)

  NEM  08:45:29
  Price=$97.80  RSI=40.1  RSI_prev=42.2
  No entry — RSI=40.1  (waiting for RSI to cross below 30)

  CTXR  08:45:30
  Price=$0.74  RSI=64.9  RSI_prev=64.9
  No entry — RSI=64.9  (waiting for RSI to cross below 30)

  ONON  08:45:31
  Price=$34.85  RSI=41.8  RSI_prev=41.8
  No entry — RSI=41.8  (waiting for RSI to cross below 30)

  IONQ  08:45:31
  Price=$31.41  RSI=48.4  RSI_prev=48.4
  OPEN  entry=$32.45  qty=40  P&L=-3.20%  ($-41.60)
  Holding — waiting for RSI > 70 to sell  (RSI=48.4)

  MARA  08:45:32
  Price=$8.95  RSI=69.3  RSI_prev=68.1
  OPEN  entry=$8.06  qty=40  P&L=+11.04%  ($+35.60)
  Holding — waiting for RSI > 70 to sell  (RSI=69.3)

  MRVL  08:45:33
  Price=$96.38  RSI=40

Error 162, reqId 94: Historical Market Data Service error message:HMDS query returned no data: CXW@SMART Trades, contract: Stock(conId=254455210, symbol='CXW', exchange='SMART', primaryExchange='NYSE', currency='USD', localSymbol='CXW', tradingClass='CXW')


...


  BRK.B  16:32:15
  No data / insufficient bars for BRK.B

  IBM  16:32:15
  Price=$242.01  RSI=44.2  RSI_prev=44.2
  OPEN  entry=$241.47  qty=40  P&L=+0.22%  ($+21.60)
  Holding — waiting for RSI > 70 to sell  (RSI=44.2)

  MSFT  16:32:16
  Price=$366.62  RSI=53.9  RSI_prev=46.4
  OPEN  entry=$368.31  qty=40  P&L=-0.46%  ($-67.60)
  Holding — waiting for RSI > 70 to sell  (RSI=53.9)

  KO  16:32:17
  Price=$74.93  RSI=46.7  RSI_prev=46.7
  No entry — RSI=46.7  (waiting for RSI to cross below 30)

  MSTR  16:32:17
  Price=$134.37  RSI=58.0  RSI_prev=55.9
  OPEN  entry=$133.95  qty=40  P&L=+0.31%  ($+16.80)
  Holding — waiting for RSI > 70 to sell  (RSI=58.0)

  COIN  16:32:18
  Price=$174.42  RSI=56.7  RSI_prev=47.5
  OPEN  entry=$180.70  qty=40  P&L=-3.48%  ($-251.20)
  Holding — waiting for RSI > 70 to sell  (RSI=56.7)

  GLD  16:32:19
  Price=$403.23  RSI=55.3  RSI_prev=57.8
  No entry — RSI=55.3  (waiting for RSI to cross below 30)

  META  16:32:19
  Price=$550.00  RSI=55.4  RSI_

Error 200, reqId 5937: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')



  SAVA  16:37:43
  No data / insufficient bars for SAVA

  SLDB  16:37:43
  Price=$7.73  RSI=41.9  RSI_prev=41.9
  No entry — RSI=41.9  (waiting for RSI to cross below 30)

  SLV  16:37:44
  Price=$62.09  RSI=63.5  RSI_prev=60.5
  OPEN  entry=$60.58  qty=40  P&L=+2.49%  ($+60.40)
  Holding — waiting for RSI > 70 to sell  (RSI=63.5)

  NEM  16:37:45
  Price=$100.24  RSI=60.4  RSI_prev=60.4
  No entry — RSI=60.4  (waiting for RSI to cross below 30)

  CTXR  16:37:45
  Price=$0.70  RSI=47.9  RSI_prev=47.9
  No entry — RSI=47.9  (waiting for RSI to cross below 30)

  ONON  16:37:46
  Price=$32.26  RSI=48.4  RSI_prev=49.7
  OPEN  entry=$34.36  qty=40  P&L=-6.11%  ($-84.00)
  Holding — waiting for RSI > 70 to sell  (RSI=48.4)

  IONQ  16:37:47
  Price=$30.30  RSI=66.1  RSI_prev=60.3
  OPEN  entry=$32.45  qty=40  P&L=-6.63%  ($-86.00)
  Holding — waiting for RSI > 70 to sell  (RSI=66.1)

  MARA  16:37:47
  Price=$8.68  RSI=57.3  RSI_prev=57.3
  No entry — RSI=57.3  (waiting for RSI to cross 

Error 200, reqId 5963: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')



  BRK.B  16:38:11
  No data / insufficient bars for BRK.B

  IBM  16:38:11
  Price=$242.01  RSI=44.2  RSI_prev=44.2
  OPEN  entry=$241.47  qty=40  P&L=+0.22%  ($+21.60)
  Holding — waiting for RSI > 70 to sell  (RSI=44.2)

  MSFT  16:38:11
  Price=$365.90  RSI=45.2  RSI_prev=49.5
  OPEN  entry=$368.31  qty=40  P&L=-0.65%  ($-96.40)
  Holding — waiting for RSI > 70 to sell  (RSI=45.2)

  KO  16:38:12
  Price=$74.93  RSI=46.7  RSI_prev=46.7
  No entry — RSI=46.7  (waiting for RSI to cross below 30)

  MSTR  16:38:13
  Price=$134.23  RSI=55.9  RSI_prev=57.8
  OPEN  entry=$133.95  qty=40  P&L=+0.21%  ($+11.20)
  Holding — waiting for RSI > 70 to sell  (RSI=55.9)

  COIN  16:38:13
  Price=$174.42  RSI=56.9  RSI_prev=54.3
  OPEN  entry=$180.70  qty=40  P&L=-3.48%  ($-251.20)
  Holding — waiting for RSI > 70 to sell  (RSI=56.9)

  GLD  16:38:14
  Price=$405.24  RSI=65.6  RSI_prev=58.9
  No entry — RSI=65.6  (waiting for RSI to cross below 30)

  META  16:38:15
  Price=$549.40  RSI=52.4  RSI_

In [ ]:
import asyncio
from ib_insync import *
import pandas as pd
import numpy as np
from datetime import datetime
import random
from pymongo import MongoClient
from bson import ObjectId
import warnings

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# DATABASE + IB CONNECTION
# ══════════════════════════════════════════════════════════════════════════════

client             = MongoClient('mongodb://localhost:27017/')
db                 = client['RSI_Strategy2']
trades_collection  = db['trades_connors_rsi2']
exclude_collection = db['excluded_tickers_connors_rsi2']

util.startLoop()
ib = IB()
ib.connect('127.0.0.1', 7497, clientId=random.randint(1, 1000))

tickers = [
    'WMT', 'MU', 'SAVA', 'SLDB', 'SLV', 'NEM', 'CTXR', 'ONON', 'IONQ', 'MARA',
    'MRVL', 'T', 'CCL', 'XYZ', 'PG', 'ONDS', 'NFLX', 'V', 'ADBE', 'MS', 'CXW',
    'BA', 'BABA', 'DAL', 'JPM', 'C', 'AMZN', 'UAL', 'BRK.B', 'IBM', 'MSFT', 'KO',
    'MSTR', 'COIN', 'GLD', 'META', 'AAL', 'OSCR', 'QSI', 'SPY', 'NVO', 'DIS',
    'AAPL', 'BMNR', 'GRAB', 'RBLX', 'AFRM', 'NVDA', 'FUBO', 'PYPL', 'GOOG', 'BTC',
    'RGTI', 'GPRO', 'OKLO', 'AMD', 'XPEV', 'SHEL', 'TSM', 'SNOW', 'NET', 'SES',
    'TSLA', 'BP', 'UBER', 'INTC', 'MRNA', 'IREN', 'ORCL', 'HIMS', 'NBIS'
]
contracts = [Stock(t, 'SMART', 'USD') for t in tickers]
ib.qualifyContracts(*contracts)


# ══════════════════════════════════════════════════════════════════════════════
# STRATEGY PARAMETERS
# ══════════════════════════════════════════════════════════════════════════════

MA_PERIOD        = 200    # Trend filter: 200-day SMA
RSI2_PERIOD      = 2      # Connors RSI period
RSI2_OVERSOLD    = 5      # Long entry: RSI(2) < 5
RSI2_OVERBOUGHT  = 95     # Short entry: RSI(2) > 95
RSI2_LONG_EXIT   = 70     # Long exit: RSI(2) > 70
RSI2_SHORT_EXIT  = 50     # Short exit: RSI(2) < 50
ORDER_SIZE       = 40     # Fixed shares per trade


# ══════════════════════════════════════════════════════════════════════════════
# ORDER HELPER  — always DAY to prevent TWS preset overriding TIF to GTC
# ══════════════════════════════════════════════════════════════════════════════

def day_market_order(action: str, qty: int) -> MarketOrder:
    """
    Return a MarketOrder with tif='DAY' explicitly set.
    This prevents TWS order presets from silently converting TIF to GTC
    (error 10349), which causes the order to be cancelled immediately.
    """
    order     = MarketOrder(action, qty)
    order.tif = 'DAY'
    return order


# ══════════════════════════════════════════════════════════════════════════════
# MONGO SANITIZER
# ══════════════════════════════════════════════════════════════════════════════

def sanitize_for_mongo(d):
    """Recursively convert numpy types to native Python types."""
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out[k] = sanitize_for_mongo(v)
        elif hasattr(v, 'item'):
            out[k] = v.item()
        else:
            out[k] = v
    return out


# ══════════════════════════════════════════════════════════════════════════════
# INDICATORS
# ══════════════════════════════════════════════════════════════════════════════

def calc_rsi(series, period=2):
    """Compute RSI for a given period on a price Series."""
    d    = series.diff()
    gain = d.clip(lower=0).ewm(com=period - 1, min_periods=period).mean()
    loss = (-d.clip(upper=0)).ewm(com=period - 1, min_periods=period).mean()
    return 100 - 100 / (1 + gain / (loss + 1e-10))


def calc_indicators(df, ma_period=MA_PERIOD, rsi_period=RSI2_PERIOD):
    """
    Add 200-day SMA (trend filter) and RSI(2) (entry/exit signal) to df.

    NOTE: The 200-day MA requires daily bars.  We request daily bars
    separately inside check_and_trade() and attach the result here.
    """
    df['RSI2'] = calc_rsi(df['close'], period=rsi_period)
    return df


# ══════════════════════════════════════════════════════════════════════════════
# DATA FETCHING
# ══════════════════════════════════════════════════════════════════════════════

def get_daily_ma(contract, period=MA_PERIOD):
    """
    Fetch enough daily bars to compute the 200-day SMA.
    Returns the latest SMA value and the latest closing price.
    """
    bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr=f'{period + 10} D',   # a little extra buffer
        barSizeSetting='1 day',
        whatToShow='TRADES',
        useRTH=True,
        formatDate=1
    )
    if not bars or len(bars) < period:
        return None, None

    df        = util.df(bars)
    df.columns = [c.lower() for c in df.columns]
    sma       = float(df['close'].rolling(period).mean().iloc[-1])
    last_close = float(df['close'].iloc[-1])
    return sma, last_close


def get_intraday_rsi2(contract):
    """
    Fetch 5-min intraday bars and compute RSI(2).
    Returns (price_now, rsi2_now, rsi2_prev) or (None, None, None).
    """
    bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr='3 D',           # 3 days of 5-min bars for warm-up
        barSizeSetting='5 mins',
        whatToShow='TRADES',
        useRTH=False,
        formatDate=1
    )
    if not bars or len(bars) < 10:
        return None, None, None

    df         = util.df(bars)
    df.columns = [c.lower() for c in df.columns]
    df         = calc_indicators(df)

    price     = float(df.iloc[-1]['close'])
    rsi2_now  = float(df.iloc[-1]['RSI2'])
    rsi2_prev = float(df.iloc[-2]['RSI2'])
    return price, rsi2_now, rsi2_prev


# ══════════════════════════════════════════════════════════════════════════════
# MONGO HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def create_trade_doc(symbol, direction, entry_price, qty, rsi2_at_entry, sma200):
    return {
        'instrument':      symbol,
        'direction':       direction,            # 'long' or 'short'
        'entry_price':     float(entry_price),
        'quantity':        int(qty),
        'remaining_qty':   int(qty),
        'rsi2_at_entry':   float(rsi2_at_entry),
        'sma200_at_entry': float(sma200),
        'entry_timestamp': datetime.now(),
        'order_id':        str(ObjectId()),
        'exit_price':      None,
        'exit_timestamp':  None,
        'exit_rsi2':       None,
        'reason':          None,
        'realized_pnl':    None,
        'status':          'open',
        'created_at':      datetime.now(),
        'updated_at':      datetime.now(),
    }


def db_update(trade_id, updates):
    updates['updated_at'] = datetime.now()
    trades_collection.update_one({'_id': ObjectId(trade_id)}, {'$set': updates})


def do_sell(contract, qty, entry_price, sell_price, rsi2, reason, trade_id):
    """Execute market sell (close long) and persist to MongoDB."""
    ib.placeOrder(contract, day_market_order('SELL', qty))
    pnl  = (sell_price - entry_price) * qty
    sign = '+' if pnl >= 0 else ''
    db_update(trade_id, {
        'exit_price':     float(sell_price),
        'exit_timestamp': datetime.now(),
        'exit_rsi2':      float(rsi2),
        'reason':         reason,
        'realized_pnl':   float(pnl),
        'status':         'closed',
    })
    print(f"  ✅ SELL (close long) {qty}sh @ ${sell_price:.2f}"
          f"  RSI2={rsi2:.1f}  [{reason}]  PnL: {sign}${pnl:.2f}")


def do_cover(contract, qty, entry_price, cover_price, rsi2, reason, trade_id):
    """Execute market buy (cover short) and persist to MongoDB."""
    ib.placeOrder(contract, day_market_order('BUY', qty))
    pnl  = (entry_price - cover_price) * qty   # short PnL is reversed
    sign = '+' if pnl >= 0 else ''
    db_update(trade_id, {
        'exit_price':     float(cover_price),
        'exit_timestamp': datetime.now(),
        'exit_rsi2':      float(rsi2),
        'reason':         reason,
        'realized_pnl':   float(pnl),
        'status':         'closed',
    })
    print(f"  ✅ COVER (close short) {qty}sh @ ${cover_price:.2f}"
          f"  RSI2={rsi2:.1f}  [{reason}]  PnL: {sign}${pnl:.2f}")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

async def check_and_trade():
    """
    ╔══════════════════════════════════════════════════════════════════════════╗
    ║  CONNORS RSI(2) MEAN-REVERSION STRATEGY                                 ║
    ╠══════════════════════════════════════════════════════════════════════════╣
    ║                                                                          ║
    ║  TREND FILTER  — 200-day SMA (daily bars)                               ║
    ║    Price > SMA200 → bullish bias  → longs only                          ║
    ║    Price < SMA200 → bearish bias  → shorts only                         ║
    ║                                                                          ║
    ║  ENTRY (5-min RSI2)                                                      ║
    ║    Long  — RSI(2) < 5   (strongly oversold dip inside uptrend)          ║
    ║    Short — RSI(2) > 95  (strongly overbought spike inside downtrend)    ║
    ║                                                                          ║
    ║  EXIT                                                                    ║
    ║    Long  — RSI(2) > 70  (mean-reversion complete)                       ║
    ║    Short — RSI(2) < 50  (mean-reversion complete)                       ║
    ║                                                                          ║
    ║  Position size: 40 shares fixed                                          ║
    ║  One trade per ticker per day                                            ║
    ║                                                                          ║
    ╚══════════════════════════════════════════════════════════════════════════╝
    """

    while True:
        ib.positions()   # refresh internal positions cache

        for contract in contracts:
            symbol = contract.symbol
            print(f"\n{'='*60}")
            print(f"  {symbol}  {datetime.now().strftime('%H:%M:%S')}")
            print(f"{'='*60}")

            # ── 1. Trend filter: 200-day SMA ──────────────────────────────
            sma200, daily_close = get_daily_ma(contract)
            if sma200 is None:
                print(f"  Insufficient daily data for SMA200 — skipping {symbol}")
                continue

            price_above_sma = daily_close > sma200
            trend_label     = 'UPTREND ▲' if price_above_sma else 'DOWNTREND ▼'
            print(f"  SMA200=${sma200:.2f}  DailyClose=${daily_close:.2f}  {trend_label}")

            # ── 2. RSI(2) on 5-min bars ────────────────────────────────────
            price, rsi2_now, rsi2_prev = get_intraday_rsi2(contract)
            if price is None:
                print(f"  Insufficient intraday data — skipping {symbol}")
                continue

            print(f"  Price=${price:.2f}  RSI2={rsi2_now:.1f}  RSI2_prev={rsi2_prev:.1f}")

            open_trade = trades_collection.find_one({'instrument': symbol, 'status': 'open'})

            # ══════════════════════════════════════════════════════════════
            # POSITION MANAGEMENT
            # ══════════════════════════════════════════════════════════════

            if open_trade:
                entry_price = float(open_trade['entry_price'])
                qty         = int(open_trade['remaining_qty'])
                direction   = open_trade.get('direction', 'long')
                trade_id    = open_trade['_id']

                if direction == 'long':
                    current_pnl = (price - entry_price) * qty
                    profit_pct  = (price - entry_price) / entry_price
                    print(f"  OPEN LONG  entry=${entry_price:.2f}  qty={qty}"
                          f"  P&L={profit_pct:+.2%}  (${current_pnl:+.2f})")

                    # Exit long when RSI(2) > 70
                    if rsi2_now > RSI2_LONG_EXIT:
                        do_sell(contract, qty, entry_price, price,
                                rsi2_now, 'RSI2_ABOVE_70_LONG_EXIT', trade_id)
                        print(f"  📈 LONG EXIT: RSI2={rsi2_now:.1f} > {RSI2_LONG_EXIT}")
                    else:
                        print(f"  Holding long — waiting RSI2 > {RSI2_LONG_EXIT}  (RSI2={rsi2_now:.1f})")

                elif direction == 'short':
                    current_pnl = (entry_price - price) * qty
                    profit_pct  = (entry_price - price) / entry_price
                    print(f"  OPEN SHORT  entry=${entry_price:.2f}  qty={qty}"
                          f"  P&L={profit_pct:+.2%}  (${current_pnl:+.2f})")

                    # Exit short when RSI(2) < 50
                    if rsi2_now < RSI2_SHORT_EXIT:
                        do_cover(contract, qty, entry_price, price,
                                 rsi2_now, 'RSI2_BELOW_50_SHORT_EXIT', trade_id)
                        print(f"  📉 SHORT EXIT: RSI2={rsi2_now:.1f} < {RSI2_SHORT_EXIT}")
                    else:
                        print(f"  Holding short — waiting RSI2 < {RSI2_SHORT_EXIT}  (RSI2={rsi2_now:.1f})")

            # ══════════════════════════════════════════════════════════════
            # ENTRY LOGIC
            # ══════════════════════════════════════════════════════════════

            else:
                today = datetime.now().date()
                if exclude_collection.find_one({'ticker': symbol, 'date': today.isoformat()}):
                    print(f"  Already traded {symbol} today — skipping.")
                    continue

                # ── LONG: uptrend + RSI(2) oversold ──────────────────────
                if price_above_sma and rsi2_now < RSI2_OVERSOLD:
                    ib.placeOrder(contract, day_market_order('BUY', ORDER_SIZE))

                    doc = create_trade_doc(symbol, 'long', price, ORDER_SIZE, rsi2_now, sma200)
                    trades_collection.insert_one(sanitize_for_mongo(doc))
                    exclude_collection.insert_one({'ticker': symbol, 'date': today.isoformat()})

                    print(f"  🚀 BUY (long)  {symbol}")
                    print(f"     Entry:  ${price:.2f}  |  SMA200=${sma200:.2f}  (above)")
                    print(f"     Shares: {ORDER_SIZE}")
                    print(f"     RSI2:   {rsi2_now:.1f}  (oversold dip < {RSI2_OVERSOLD})")

                # ── SHORT: downtrend + RSI(2) overbought ─────────────────
                elif not price_above_sma and rsi2_now > RSI2_OVERBOUGHT:
                    ib.placeOrder(contract, day_market_order('SELL', ORDER_SIZE))

                    doc = create_trade_doc(symbol, 'short', price, ORDER_SIZE, rsi2_now, sma200)
                    trades_collection.insert_one(sanitize_for_mongo(doc))
                    exclude_collection.insert_one({'ticker': symbol, 'date': today.isoformat()})

                    print(f"  🔻 SELL (short)  {symbol}")
                    print(f"     Entry:  ${price:.2f}  |  SMA200=${sma200:.2f}  (below)")
                    print(f"     Shares: {ORDER_SIZE}")
                    print(f"     RSI2:   {rsi2_now:.1f}  (overbought spike > {RSI2_OVERBOUGHT})")

                else:
                    bias = 'long' if price_above_sma else 'short'
                    threshold = f'< {RSI2_OVERSOLD}' if price_above_sma else f'> {RSI2_OVERBOUGHT}'
                    print(f"  No entry — bias={bias}  RSI2={rsi2_now:.1f}"
                          f"  (waiting for RSI2 {threshold})")

            await asyncio.sleep(0.5)   # brief pause between tickers

        print(f"\n{'='*60}")
        print(f"  Scan complete. Next scan in 5 minutes.")
        print(f"{'='*60}")
        await asyncio.sleep(300)


# ══════════════════════════════════════════════════════════════════════════════
# RUN
# ══════════════════════════════════════════════════════════════════════════════

try:
    ib.run(check_and_trade())
except Exception as e:
    print(f"Error: {e}")
    import traceback; traceback.print_exc()
finally:
    ib.disconnect()
    print("Disconnected from IB.")


Error 200, reqId 5: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')
Error 200, reqId 31: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')
Unknown contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')
Unknown contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')



  WMT  19:03:53
  SMA200=$108.54  DailyClose=$122.18  UPTREND ▲
  Price=$122.68  RSI2=99.9  RSI2_prev=99.9
  No entry — bias=long  RSI2=99.9  (waiting for RSI2 < 5)

  MU  19:03:54
  SMA200=$237.25  DailyClose=$355.46  UPTREND ▲
  Price=$356.71  RSI2=56.7  RSI2_prev=5.7
  No entry — bias=long  RSI2=56.7  (waiting for RSI2 < 5)


Error 200, reqId 78: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')



  SAVA  19:03:55
  Insufficient daily data for SMA200 — skipping SAVA

  SLDB  19:03:55
  SMA200=$5.79  DailyClose=$7.73  UPTREND ▲
  Price=$7.72  RSI2=0.0  RSI2_prev=0.3
  OPEN LONG  entry=$7.73  qty=40  P&L=-0.13%  ($-0.40)
  Holding long — waiting RSI2 > 70  (RSI2=0.0)

  SLV  19:03:56
  SMA200=$52.02  DailyClose=$60.77  UPTREND ▲
  Price=$62.30  RSI2=47.6  RSI2_prev=96.2
  No entry — bias=long  RSI2=47.6  (waiting for RSI2 < 5)

  NEM  19:03:57
  SMA200=$88.83  DailyClose=$99.36  UPTREND ▲
  Price=$100.25  RSI2=88.9  RSI2_prev=9.0
  No entry — bias=long  RSI2=88.9  (waiting for RSI2 < 5)

  CTXR  19:03:58
  SMA200=$1.16  DailyClose=$0.69  DOWNTREND ▼
  Price=$0.72  RSI2=100.0  RSI2_prev=100.0
  OPEN SHORT  entry=$0.72  qty=40  P&L=+0.00%  ($+0.00)
  Holding short — waiting RSI2 < 50  (RSI2=100.0)

  ONON  19:03:59
  SMA200=$45.56  DailyClose=$32.11  DOWNTREND ▼
  Price=$32.23  RSI2=2.0  RSI2_prev=2.0
  No entry — bias=short  RSI2=2.0  (waiting for RSI2 > 95)

  IONQ  19:04:00
  SM

Error 200, reqId 129: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')


...

  Price=$93.16  RSI2=37.3  RSI2_prev=71.3
  Already traded UAL today — skipping.

  BRK.B  22:06:01
  Insufficient daily data for SMA200 — skipping BRK.B

  IBM  22:06:01
  SMA200=$278.52  DailyClose=$241.67  DOWNTREND ▼
  Price=$242.40  RSI2=100.0  RSI2_prev=100.0
  OPEN SHORT  entry=$242.28  qty=40  P&L=-0.05%  ($-4.80)
  Holding short — waiting RSI2 < 50  (RSI2=100.0)

  MSFT  22:06:02
  SMA200=$479.65  DailyClose=$365.97  DOWNTREND ▼
  Price=$366.76  RSI2=88.2  RSI2_prev=61.3
  No entry — bias=short  RSI2=88.2  (waiting for RSI2 > 95)

  KO  22:06:03
  SMA200=$71.18  DailyClose=$74.69  UPTREND ▲
  Price=$74.90  RSI2=89.9  RSI2_prev=32.0
  Already traded KO today — skipping.

  MSTR  22:06:03
  SMA200=$260.86  DailyClose=$132.93  DOWNTREND ▼
  Price=$133.13  RSI2=8.9  RSI2_prev=42.4
  No entry — bias=short  RSI2=8.9  (waiting for RSI2 > 95)

  COIN  22:06:04
  SMA200=$282.38  DailyClose=$173.38  DOWNTREND ▼
  Price=$173.79  RSI2=95.4  RSI2_prev=84.3
  OPEN SHORT  entry=$173.79  qty=

Error 200, reqId 4597: No security definition has been found for the request, contract: Stock(symbol='SAVA', exchange='SMART', currency='USD')


  Price=$356.52  RSI2=31.7  RSI2_prev=24.2
  Already traded MU today — skipping.

  SAVA  22:11:32
  Insufficient daily data for SMA200 — skipping SAVA

  SLDB  22:11:32
  SMA200=$5.79  DailyClose=$7.73  UPTREND ▲
  Price=$7.72  RSI2=0.0  RSI2_prev=0.0
  OPEN LONG  entry=$7.73  qty=40  P&L=-0.13%  ($-0.40)
  Holding long — waiting RSI2 > 70  (RSI2=0.0)

  SLV  22:11:33
  SMA200=$52.02  DailyClose=$60.77  UPTREND ▲
  Price=$62.14  RSI2=33.1  RSI2_prev=55.2
  Already traded SLV today — skipping.

  NEM  22:11:33
  SMA200=$88.83  DailyClose=$99.36  UPTREND ▲
  Price=$100.23  RSI2=93.6  RSI2_prev=93.6
  No entry — bias=long  RSI2=93.6  (waiting for RSI2 < 5)

  CTXR  22:11:34
  SMA200=$1.16  DailyClose=$0.69  DOWNTREND ▼
  Price=$0.69  RSI2=0.0  RSI2_prev=0.0
  Already traded CTXR today — skipping.

  ONON  22:11:34
  SMA200=$45.56  DailyClose=$32.11  DOWNTREND ▼
  Price=$32.23  RSI2=88.9  RSI2_prev=88.9
  No entry — bias=short  RSI2=88.9  (waiting for RSI2 > 95)

  IONQ  22:11:35
  SMA200

Error 200, reqId 4648: No security definition has been found for the request, contract: Stock(symbol='BRK.B', exchange='SMART', currency='USD')


  Price=$93.16  RSI2=37.3  RSI2_prev=71.3
  Already traded UAL today — skipping.

  BRK.B  22:11:50
  Insufficient daily data for SMA200 — skipping BRK.B

  IBM  22:11:50
  SMA200=$278.52  DailyClose=$241.67  DOWNTREND ▼
  Price=$242.40  RSI2=100.0  RSI2_prev=100.0
  OPEN SHORT  entry=$242.28  qty=40  P&L=-0.05%  ($-4.80)
  Holding short — waiting RSI2 < 50  (RSI2=100.0)

  MSFT  22:11:51
  SMA200=$479.65  DailyClose=$365.97  DOWNTREND ▼
  Price=$366.76  RSI2=88.2  RSI2_prev=61.3
  No entry — bias=short  RSI2=88.2  (waiting for RSI2 > 95)

  KO  22:11:52
  SMA200=$71.18  DailyClose=$74.69  UPTREND ▲
  Price=$74.90  RSI2=89.9  RSI2_prev=32.0
  Already traded KO today — skipping.

  MSTR  22:11:52
  SMA200=$260.86  DailyClose=$132.93  DOWNTREND ▼
  Price=$133.13  RSI2=8.9  RSI2_prev=42.4
  No entry — bias=short  RSI2=8.9  (waiting for RSI2 > 95)

  COIN  22:11:53
  SMA200=$282.38  DailyClose=$173.38  DOWNTREND ▼
  Price=$173.79  RSI2=95.4  RSI2_prev=84.3
  OPEN SHORT  entry=$173.79  qty=